In [2]:
import math
import torch
import random
import numpy as np
from torch import nn, optim
import json

from collections import defaultdict
from datetime import datetime
from dataclasses import dataclass, asdict
from tokenizer import TinyStoriesTokenizer

from torch.utils.data import Sampler, SubsetRandomSampler
from torch.utils.data import Dataset, DataLoader
from tokenizer import TinyStoriesTokenizer

# ============= Hyper-parameters for training ============== #

@dataclass
class Config :
    vocab_size: int = 5000  # This number should agree with the tokenizer
    number_of_transformer_blocks: int = 4
    number_of_attention_heads: int = 1
    vector_dim: int = 256
    block_size: int = 512
    dropout_prob: float = 0.1
    batch_size: int = 8
    learning_rate: float = 0.0005
    weight_decay: float = 0.000001
    no_of_epochs: int = 1


class TinyStoriesDataset(Dataset):
    def __init__(self, data_file, block_size):
        """
        data_file: path to the .bin file (uint16 array of token IDs)
        block_size: the context window (e.g., 256 or 512 tokens)
        """

        # Memory-map the data file (RAM usage stays near zero!)
        self.data = np.memmap(data_file, dtype=np.uint16, mode='r')
        self.block_size = block_size

    def __len__(self):
        # We subtract block_size to ensure we don't go out of bounds
        return len(self.data) - self.block_size

    def __getitem__(self, idx):
        # Pull a chunk of length block_size + 1 (data and target)
        chunk = self.data[idx : idx + self.block_size + 1]

        # Convert to torch tensors
        x = torch.from_numpy(chunk[:-1].astype(np.int64)) # Input
        y = torch.from_numpy(chunk[1:].astype(np.int64))  # Target (shifted by 1)

        return x, y 




In [3]:
!pip install spacy
!python -m spacy download en_core_web_sm
import spacy

  Using cached spacy-3.8.14-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (28 kB)
  Using cached spacy_legacy-3.0.12-py2.py3-none-any.whl.metadata (2.8 kB)
  Using cached spacy_loggers-1.0.5-py3-none-any.whl.metadata (23 kB)
  Using cached murmurhash-1.0.15-cp311-cp311-manylinux1_x86_64.manylinux2014_x86_64.manylinux_2_17_x86_64.manylinux_2_5_x86_64.whl.metadata (2.3 kB)
  Using cached cymem-2.0.13-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (9.7 kB)
  Using cached preshed-3.0.13-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (5.2 kB)
  Using cached thinc-8.3.13-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (14 kB)
  Using cached wasabi-1.1.3-py3-none-any.whl.metadata (28 kB)
  Using cached srsly-2.5.3-cp311-cp311-manylinux2014_x86_64.manylinux_2_17_x86_64.whl.metadata (19 kB)
  Using cached catalogue-2.0.10-py3-none-any.whl.metadata (14 kB)
  Using cached weasel-1.0.0-py3-none-any.whl.metadata (4.

In [4]:
def align_tags_with_bpe(tokens_bpe, tokens_spacy, tags_spacy):
    bpe_tags_aligned = []
    idx_spacy = 0  # recorrer word spacy

    acumulado_bpe = ""
    acumulado_spacy = ""
    tag_actual = None
    
    for token in tokens_bpe:
        token_limpio = token.replace(" ", "")
        acumulado_bpe += token_limpio
        
        if not token_limpio:
            bpe_tags_aligned.append("X")
            continue

        is_first_subtoken = False
            
        # If our spacy separates token in 2 dif. Then bpe_acum will be ahead spacy
        while len(acumulado_bpe) > len(acumulado_spacy) and idx_spacy < len(tokens_spacy):
            acumulado_spacy += tokens_spacy[idx_spacy].replace(" ", "")
            tag_actual = tags_spacy[idx_spacy]
            idx_spacy += 1
            # if in -> this tokes is the begining or whole
            is_first_subtoken = True

        if is_first_subtoken:
            bpe_tags_aligned.append(tag_actual)
        else:
            bpe_tags_aligned.append("SUBWORD")
        
    return bpe_tags_aligned

def tensor_tags(ids_tokens_dataset, tokenizer, nlp, tag2id):
    tokens_bpe = tokenizer.decode_to_tokens(ids_tokens_dataset)
    text = tokenizer.decode(ids_tokens_dataset)
    
    # SpaCy
    doc = nlp(text)
    tokens_spacy = [t.text for t in doc]
    tags_spacy = [t.pos_ for t in doc]
    tags_aligned = align_tags_with_bpe(tokens_bpe, tokens_spacy, tags_spacy)
    
    # strings tags to id
    # if no tag - None - 'X'
    tags_ids = []
    for tag in tags_aligned:
        if tag in tag2id:
            tags_ids.append(tag2id[tag])
        else:
            tags_ids.append(tag2id['X'])

    return torch.tensor(tags_ids, dtype=torch.long)

In [5]:
import random 

POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}
id2tag = {idx: tag for tag, idx in tag2id.items()}
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)

print(f"Dataset len: {len(training_dataset)} stories")

tensores_tags_dataset = torch.load('tags_probes_5k_first_subtoken.pt')

idx_aleatorio = random.randint(0, len(tensores_tags_dataset) - 1)
tensor_tags_elegido = tensores_tags_dataset[idx_aleatorio]

# 3. Recuperar el texto BPE de esa misma historia para poder contrastarlo
ejemplo_original = training_dataset[idx_aleatorio]
tokens_bpe_reales = tokenizer.decode_to_tokens(ejemplo_original[0].tolist())


Dataset len: 18507842 stories


In [6]:
# Check tags to see correct mapping
todos_los_ids_presentes = set()

for i in range(10):
    ids_unicos_historia = tensores_tags_dataset[i].tolist()
    todos_los_ids_presentes.update(ids_unicos_historia)

print("id num in tensors (all ids found):")
print(todos_los_ids_presentes)

print("\n mapping tag2id")
print(tag2id)


id num in tensors (all ids found):
{1, 3, 4, 5, 6, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 19}

 mapping tag2id
{'PAD': 0, 'SUBWORD': 1, 'X': 2, 'ADJ': 3, 'ADP': 4, 'ADV': 5, 'AUX': 6, 'CONJ': 7, 'CCONJ': 8, 'DET': 9, 'INTJ': 10, 'NOUN': 11, 'NUM': 12, 'PART': 13, 'PRON': 14, 'PROPN': 15, 'PUNCT': 16, 'SCONJ': 17, 'SYM': 18, 'VERB': 19}


In [7]:
set_pos = defaultdict(set)
count = 0
for idx in range(len(tensores_tags_dataset)):    
    tags = tensores_tags_dataset[idx]
    ids = training_dataset[idx][0].tolist()
    tokens = tokenizer.decode_to_tokens(ids)
    
    for i, tid in enumerate(tags.tolist()):
        tok_txt = tokens[i].replace('\n', '\\n')
        tag_name = id2tag[tid]
        if tid <= 2:
            pass
        else:
            set_pos[tok_txt].add(tid)

# dictionary: [word : [POS_TAG_ID_1, POS_TAG_ID_2,...]
dict_multipos =  {}
for word_set, tag_set in set_pos.items():
    if len(tag_set) > 1:
        dict_multipos[word_set] = list(tag_set)

print(dict_multipos)

{' Jack': [11, 15], ' was': [19, 6], ' playing': [11, 19], ' with': [17, 4], '.': [16, 11, 15], ' to': [4, 13], ' zoom': [11, 19], ' around': [4, 5], ' the': [9, 14], ' this': [9, 14], ' scarf': [19, 11], '!': [16, 18, 15], ' had': [19, 6], ' in': [4, 5], ' he': [19, 14], ' so': [17, 5], ' walk': [11, 19], ' back': [4, 5], ' before': [17, 4, 5], ' a': [9, 14], ' loud': [3, 5], ' up': [4, 5], ' neigh': [11, 15], ' her': [11, 14], ' sun': [19, 11], ' have': [19, 6], '"': [16, 11, 12], ' face': [19, 11], ' look': [11, 19], ' as': [17, 4, 5], ' po': [11, 3], ' close': [3, 5], ' upon': [17, 5], ' there': [5, 14], ' boy': [11, 15], ' The': [9, 14], ' scared': [19, 3], ' for': [17, 4], ' mommy': [11, 15], ' came': [19, 6], ' surprised': [19, 3], ' that': [9, 17, 14], ' on': [4, 5], ' were': [19, 6], ' best': [3, 5], ' an': [9, 14], ' fast': [3, 19, 5], ' sneeze': [11, 19], ' hard': [3, 5], ' all': [9, 5, 14], ' at': [4, 5], 'ite': [19, 15], ' down': [4, 5], ' home': [11, 5], ' much': [3, 5], 

In [16]:
with open("multipos_dict.json", "w", encoding="utf-8") as f:
    json.dump(dict_multipos, f)

with open('multipos_dict.json', 'r') as f:
    dict_multipos = json.load(f)

In [6]:
idx = 0    
tags = tensores_tags_dataset[idx]
ids = training_dataset[idx][0].tolist()
tokens = tokenizer.decode_to_tokens(ids)

multipos_dict = {}

for i, tid in enumerate(tags.tolist()):    
    tok_txt = tokens[i].replace('\n', '\\n')
    tag_name = id2tag[tid]
    print(f"Story: '{idx}'|Token_id: '{i}' | Token: '{tok_txt}' | tag: {tid} ({tag_name})")



Story: '0'|Token_id: '0' | Token: 'One' | tag: 12 (NUM)
Story: '0'|Token_id: '1' | Token: ' day' | tag: 11 (NOUN)
Story: '0'|Token_id: '2' | Token: ',' | tag: 16 (PUNCT)
Story: '0'|Token_id: '3' | Token: ' Jack' | tag: 15 (PROPN)
Story: '0'|Token_id: '4' | Token: ' was' | tag: 6 (AUX)
Story: '0'|Token_id: '5' | Token: ' playing' | tag: 19 (VERB)
Story: '0'|Token_id: '6' | Token: ' with' | tag: 4 (ADP)
Story: '0'|Token_id: '7' | Token: ' his' | tag: 14 (PRON)
Story: '0'|Token_id: '8' | Token: ' favour' | tag: 3 (ADJ)
Story: '0'|Token_id: '9' | Token: 'ite' | tag: 1 (SUBWORD)
Story: '0'|Token_id: '10' | Token: ' red' | tag: 3 (ADJ)
Story: '0'|Token_id: '11' | Token: ' car' | tag: 11 (NOUN)
Story: '0'|Token_id: '12' | Token: '.' | tag: 16 (PUNCT)
Story: '0'|Token_id: '13' | Token: ' He' | tag: 14 (PRON)
Story: '0'|Token_id: '14' | Token: ' loved' | tag: 19 (VERB)
Story: '0'|Token_id: '15' | Token: ' to' | tag: 13 (PART)
Story: '0'|Token_id: '16' | Token: ' zoom' | tag: 19 (VERB)
Story: '0

In [8]:
#alto

In [9]:
"""
inicio_muestra = 492
fin_muestra = 512
print('id historia:', idx_aleatorio)
for i in range(inicio_muestra, fin_muestra):
    token_texto = tokens_bpe_reales[i].replace('\n', '\\n') 
    tag_id = tensor_tags_elegido[i].item()  # tensor num
    tag_texto = id2tag[tag_id]  #to string
    
    print(f"  [{i}] Token: {token_texto:15} ----> Tag en Tensor: {tag_texto}")
  """  

'\ninicio_muestra = 492\nfin_muestra = 512\nprint(\'id historia:\', idx_aleatorio)\nfor i in range(inicio_muestra, fin_muestra):\n    token_texto = tokens_bpe_reales[i].replace(\'\n\', \'\\n\') \n    tag_id = tensor_tags_elegido[i].item()  # tensor num\n    tag_texto = id2tag[tag_id]  #to string\n    \n    print(f"  [{i}] Token: {token_texto:15} ----> Tag en Tensor: {tag_texto}")\n  '

In [10]:
alto

#empieza procesamiento tags

NameError: name 'alto' is not defined

In [ ]:
import os
import torch
import spacy
from tqdm import tqdm

POS_TAGS_VOCAB = [
    'PAD', 'SUBWORD', 'X', 'ADJ', 'ADP', 'ADV', 'AUX', 'CONJ', 
    'CCONJ', 'DET', 'INTJ', 'NOUN', 'NUM', 'PART', 'PRON', 
    'PROPN', 'PUNCT', 'SCONJ', 'SYM', 'VERB'
]
tag2id = {tag: idx for idx, tag in enumerate(POS_TAGS_VOCAB)}

nlp = spacy.load("en_core_web_sm")
tokenizer = TinyStoriesTokenizer.load('/datasets/dd2417/tokenizer.json')
training_dataset = TinyStoriesDataset('/datasets/dd2417/train.bin', Config.block_size)

# Output configurations
FINAL_FILE = 'tags_probes_5k_first_subtoken.pt'
NUM_PROBE_STORIES = 5000 #


dataset_tag_tensors = []

# 3. Resume progress from checkpoint if available
if os.path.exists(FINAL_FILE):
    print(f"Existing file detected. Loading previous progress...")
    dataset_tag_tensors = torch.load(FINAL_FILE)
    processed_stories = len(dataset_tag_tensors)
    print(f"Already have {processed_stories} stories ready.")
else:
    processed_stories = 0
    print("Starting preprocessing from scratch...")

# 4. Main preprocessing loop
if processed_stories < NUM_PROBE_STORIES:
    print(f"Processing stories from {processed_stories} to {NUM_PROBE_STORIES}:")
    
    # Loop from last checkpoint to handle failures smoothly
    for idx in tqdm(range(processed_stories, NUM_PROBE_STORIES)):
        try:
            example = training_dataset[idx]
            native_ids_list = example[0].tolist()
            
            # Alignment function
            tag_tensor = tensor_tags(native_ids_list, tokenizer, nlp, tag2id)
            dataset_tag_tensors.append(tag_tensor)
            
            # Checkpoint 
            if (idx + 1) % 500 == 0:
                torch.save(dataset_tag_tensors, FINAL_FILE)
                
        except Exception as e:
            print(f"\n Error processing story index {idx}: {e}")
            print("Saving current progress before aborting...")
            torch.save(dataset_tag_tensors, FINAL_FILE)
            raise e

    torch.save(dataset_tag_tensors, FINAL_FILE)
    print(f"saved with {NUM_PROBE_STORIES} aligned tags.")
else:
    print("\nAll 5000 stories were already fully preprocessed.")

In [ ]:


# Usaremos un defaultdict de sets para evitar duplicados
word_pos_dict = defaultdict(set)

print("Construyendo diccionario de palabras y sus POS tags...")
# Usar solo un subconjunto o todo el dataset de los tensores ya procesados
for idx in range(len(tensores_tags_dataset)):
    tags = tensores_tags_dataset[idx].tolist()
    ids = training_dataset[idx][0].tolist()
    tokens = tokenizer.decode_to_tokens(ids)

    for tok, tag_id in zip(tokens, tags):
        # Ignorar 0 (PAD), 1 (SUBWORD), 2 (X) - Solo queremos clases reales
        if tag_id >= 3:
            # Limpiamos el token de espacios o saltos de linea
            clean_tok = tok.replace(' ', '').replace('\n', '').strip()
            
            # Solo guardamos si el token no quedó vacío
            if clean_tok: 
                tag_name = id2tag[tag_id]
                word_pos_dict[clean_tok].add(tag_name)

# Convertir sets a listas para que sea compatible con JSON
word_pos_dict_serializable = {word: list(tags) for word, tags in word_pos_dict.items()}

# Guardamos en un archivo
with open('word_to_pos_dict.json', 'w', encoding='utf-8') as f:
    json.dump(word_pos_dict_serializable, f, indent=4)

print("Diccionario guardado exitosamente en 'word_to_pos_dict.json'")

# Opcional: Mostrar algunas palabras que tienen múltiples tags (ambiguas)
ambiguous_words = {w: tags for w, tags in word_pos_dict_serializable.items() if len(tags) > 1}
print(f"Total de palabras ambiguas encontradas: {len(ambiguous_words)}")
print("Ejemplo de ambiguas:", list(ambiguous_words.items())[:5])